## 📦 Imports

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import sys
import time
from itertools import combinations
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import faiss
from concurrent.futures import ThreadPoolExecutor, as_completed

sys.path.append('../src')
from ingest import normalize_to_ingredient_rxcui

## 📂 Load Knowledge Base

In [2]:
kb_df = pd.read_csv('../data/processed_interactions_kb.csv')
kb_df['pair_key'] = kb_df['pair_key'].apply(eval)

print(f"✅ Loaded knowledge base:")
print(f"   Total interactions: {len(kb_df):,}")
print(f"   Unique drug pairs: {kb_df['pair_key'].nunique():,}")
print(f"\nSample:")
print(kb_df[['Drug 1', 'Drug 2', 'Interaction Description', 'pair_key']].head())

✅ Loaded knowledge base:
   Total interactions: 170,782
   Unique drug pairs: 170,782

Sample:
                Drug 1       Drug 2  \
0           Trioxsalen  Verteporfin   
1  Aminolevulinic acid  Verteporfin   
2     Titanium dioxide  Verteporfin   
3     Tiaprofenic acid  Verteporfin   
4          Cyamemazine  Verteporfin   

                             Interaction Description          pair_key  
0  Trioxsalen may increase the photosensitizing a...   (10844, 118886)  
1  Aminolevulinic acid may increase the photosens...     (118886, 683)  
2  Titanium dioxide may increase the photosensiti...   (118886, 38323)  
3  Tiaprofenic acid may increase the photosensiti...  (118886, 618442)  
4  Cyamemazine may increase the photosensitizing ...   (118886, 21877)  


## 📂 Load RxNorm Mappings

In [3]:
with open('../data/rxnorm_lookups.pkl', 'rb') as f:
    lookups = pickle.load(f)

name_to_rxcui = lookups['name_to_rxcui']
rxcui_to_names = lookups['rxcui_to_names']
bn_to_in_map = lookups['bn_to_in_map']
ingredient_name_to_rxcui = lookups['ingredient_name_to_rxcui']

print(f"✅ Loaded RxNorm mappings:")
print(f"   Name→RxCUI: {len(name_to_rxcui):,}")
print(f"   RxCUI→Names: {len(rxcui_to_names):,}")
print(f"   Brand→Ingredient: {len(bn_to_in_map):,}")
print(f"   Ingredient cache: {len(ingredient_name_to_rxcui):,}")

✅ Loaded RxNorm mappings:
   Name→RxCUI: 157,972
   RxCUI→Names: 82,134
   Brand→Ingredient: 77,518
   Ingredient cache: 6,409


## 🔧 Drug Normalization Function

In [4]:
def normalize_drug(drug_name):
    """Normalize drug name to ingredient RxCUI using ingestion logic"""
    rxcui, dtype = normalize_to_ingredient_rxcui(
        drug_name,
        name_to_rxcui,
        bn_to_in_map,
        ingredient_name_to_rxcui
    )
    return rxcui


print("🔍 Normalization tests:")
test_drugs = ['Tylenol', 'Acetaminophen', 'Advil', 'Ibuprofen', 'Warfarin']
for drug in test_drugs:
    rxcui = normalize_drug(drug)
    print(f"   {drug:15} → RxCUI {rxcui}")

🔍 Normalization tests:
   Tylenol         → RxCUI 161
   Acetaminophen   → RxCUI 161
   Advil           → RxCUI 5640
   Ibuprofen       → RxCUI 5640
   Warfarin        → RxCUI 11289


## 🧠 Build or Load FAISS Index

In [5]:
load_dotenv("../.env")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

index_path = Path("../data/faiss_index.bin")
embeddings_path = Path("../data/interaction_embeddings.npy")


def get_openai_embeddings_batch(texts, model="text-embedding-3-large"):
    """Generate embeddings with retry logic and latency tracking"""
    for attempt in range(3):
        try:
            t0 = time.time()
            resp = client.embeddings.create(model=model, input=texts)
            latency = (time.time() - t0) * 1000
            print(f"Batch of {len(texts)} | Latency: {latency:.1f} ms")
            embeddings = [d.embedding for d in resp.data]
            return np.array(embeddings, dtype="float32")
        except Exception as e:
            print(f"⚠️ Retry {attempt + 1}/3 for batch after error: {e}")
            time.sleep(2)
    raise RuntimeError("Failed to embed batch after 3 retries.")


if index_path.exists() and embeddings_path.exists():
    print("📂 Loading existing FAISS index...")
    index = faiss.read_index(str(index_path))
    embeddings = np.load(str(embeddings_path))
    print(f"✅ Loaded index with {index.ntotal:,} vectors ({embeddings.shape[1]} dims)")
else:
    print("🔨 Building FAISS index (first run)...")
    texts = [
        f"{row['Drug 1']} and {row['Drug 2']} interaction: {row['Interaction Description']}"
        for _, row in kb_df.iterrows()
    ]
    print(f"Generating embeddings for {len(texts):,} interactions...")

    batch_size = 1000
    all_batches = [texts[i:i + batch_size] for i in range(0, len(texts), batch_size)]
    start = time.time()

    all_embeddings = []
    with ThreadPoolExecutor(max_workers=5) as executor:
        futures = {executor.submit(get_openai_embeddings_batch, batch): idx
                   for idx, batch in enumerate(all_batches, 1)}
        for future in as_completed(futures):
            batch_idx = futures[future]
            try:
                batch_embeddings = future.result()
                all_embeddings.append(batch_embeddings)
                print(f"✅ Completed batch {batch_idx}/{len(all_batches)}")

                if len(all_embeddings) % 5 == 0:
                    temp_embeddings = np.vstack(all_embeddings)
                    np.save(embeddings_path.with_suffix(".tmp.npy"), temp_embeddings)
                    print(f"💾 Checkpoint saved ({len(temp_embeddings):,} embeddings)")
            except Exception as e:
                print(f"❌ Batch {batch_idx} failed: {e}")

    embeddings = np.vstack(all_embeddings)
    print(f"✅ Generated {embeddings.shape[0]:,} embeddings ({embeddings.shape[1]} dims)")

    embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    index_path.parent.mkdir(exist_ok=True)
    faiss.write_index(index, str(index_path))
    np.save(str(embeddings_path), embeddings)
    total_min = (time.time() - start) / 60
    print(f"✅ Saved FAISS index ({index.ntotal:,} vectors) in {total_min:.1f} min")

📂 Loading existing FAISS index...
✅ Loaded index with 170,782 vectors (3072 dims)


## 🎯 Tier 1: Direct KB Lookup

In [6]:
def direct_lookup_dynamic(drug_input):
    """Direct KB lookup via RxCUI pair matching"""
    queries = [drug_input] if isinstance(drug_input, str) else drug_input
    all_results = []

    for query_text in queries:
        clean_query = query_text.lower().replace("interaction", "").strip()

        if " and " in clean_query:
            parts = clean_query.split(" and ")
            drug1 = parts[0].strip()
            drug2 = parts[1].strip().split()[0]
        else:
            print(f"   ⚠️ Invalid format: {query_text}")
            all_results.append({"query": query_text, "hits": [], "tier": 1})
            continue

        rxcui1 = normalize_drug(drug1)
        rxcui2 = normalize_drug(drug2)

        if not rxcui1 or not rxcui2:
            print(f"   ⚠️ Could not normalize: {drug1} or {drug2}")
            all_results.append({"query": query_text, "hits": [], "tier": 1})
            continue

        pair_key = tuple(sorted([rxcui1, rxcui2]))
        match = kb_df[kb_df['pair_key'] == pair_key]

        hits = []
        if not match.empty:
            hits.append({
                'drug1': match['Drug 1'].values[0],
                'drug2': match['Drug 2'].values[0],
                'evidence': match['Interaction Description'].values[0],
                'retrieval_score': 1.0
            })
            print(f"Direct match: {drug1} + {drug2}")
        else:
            print(f"No direct match: {drug1} + {drug2}")

        all_results.append({"query": query_text, "hits": hits, "tier": 1})

    return all_results

## 🔎 Tier 2: FAISS Semantic Search

In [7]:
def semantic_search_dynamic(query_input, k=5):
    """Semantic search with lexical pre-filtering"""
    CONFIDENCE_THRESHOLD = 0.6
    FILTER_SIZE_LIMIT = 1000

    queries = [query_input] if isinstance(query_input, str) else query_input
    all_results = []

    for query_text in queries:
        response = client.embeddings.create(
            model="text-embedding-3-large",
            input=[query_text]
        )
        query_embedding = np.array([response.data[0].embedding], dtype='float32')
        query_embedding /= np.linalg.norm(query_embedding, axis=1, keepdims=True) + 1e-8

        mask = None
        if " and " in query_text.lower():
            parts = query_text.split(" and ")
            drug1 = parts[0].strip()
            drug2 = parts[1].split()[0].strip()
            mask = (
                    kb_df['Drug 1'].str.contains(drug1, case=False, na=False) |
                    kb_df['Drug 2'].str.contains(drug1, case=False, na=False) |
                    kb_df['Drug 1'].str.contains(drug2, case=False, na=False) |
                    kb_df['Drug 2'].str.contains(drug2, case=False, na=False)
            )

        if mask is not None and mask.sum() > 0 and mask.sum() <= FILTER_SIZE_LIMIT:
            filtered_df = kb_df[mask]
            subset_embeddings = embeddings[filtered_df.index.values]
            index_subset = faiss.IndexFlatIP(subset_embeddings.shape[1])
            index_subset.add(subset_embeddings)
        else:
            filtered_df = kb_df
            index_subset = index

        distances, indices = index_subset.search(query_embedding, k)
        similarities = distances[0]

        hits = []
        for idx, score in zip(indices[0], similarities):
            if score >= CONFIDENCE_THRESHOLD:
                row = filtered_df.iloc[idx]
                hits.append({
                    "drug1": row['Drug 1'],
                    "drug2": row['Drug 2'],
                    "evidence": row['Interaction Description'],
                    "retrieval_score": float(score)
                })

        all_results.append({"query": query_text, "hits": hits, "tier": 2})

    return all_results

## 🔄 Unified Hybrid Retrieval (Tier 1 → Tier 2 → Tier 3)

In [8]:
def check_drug_interaction(drug_input, k=5):
    """Tiered retrieval: direct lookup → semantic search → no evidence"""
    CONFIDENCE_THRESHOLD = 0.6
    FILTER_SIZE_LIMIT = 1000

    queries = [drug_input] if isinstance(drug_input, str) else drug_input
    all_results = []

    for query_text in queries:
        clean_query = query_text.lower().replace("interaction", "").strip()

        if " and " not in clean_query:
            print(f"   ⚠️ Invalid format: {query_text}")
            all_results.append({"query": query_text, "hits": [], "tier": 3})
            continue

        parts = clean_query.split(" and ")
        drug1 = parts[0].strip()
        drug2 = parts[1].strip().split()[0]

        rxcui1 = normalize_drug(drug1)
        rxcui2 = normalize_drug(drug2)

        if rxcui1 and rxcui2:
            pair_key = tuple(sorted([rxcui1, rxcui2]))
            match = kb_df[kb_df['pair_key'] == pair_key]

            if not match.empty:
                hits = [{
                    'drug1': match['Drug 1'].values[0],
                    'drug2': match['Drug 2'].values[0],
                    'evidence': match['Interaction Description'].values[0],
                    'retrieval_score': 1.0
                }]
                print(f"Tier 1: Direct match for {drug1} + {drug2}")
                all_results.append({"query": query_text, "hits": hits, "tier": 1})
                continue
            else:
                print(f"Tier 1: No direct match, trying semantic search...")
        else:
            print(f"   ⚠️ Could not normalize via RxCUI, trying semantic search...")

        print(f"Tier 2: Semantic search for {drug1} + {drug2}...")

        response = client.embeddings.create(
            model="text-embedding-3-large",
            input=[f"{drug1} and {drug2} interaction"]
        )
        query_embedding = np.array([response.data[0].embedding], dtype='float32')
        query_embedding /= np.linalg.norm(query_embedding, axis=1, keepdims=True) + 1e-8

        mask = (
                kb_df['Drug 1'].str.contains(drug1, case=False, na=False) |
                kb_df['Drug 2'].str.contains(drug1, case=False, na=False) |
                kb_df['Drug 1'].str.contains(drug2, case=False, na=False) |
                kb_df['Drug 2'].str.contains(drug2, case=False, na=False)
        )

        if mask.sum() > 0 and mask.sum() <= FILTER_SIZE_LIMIT:
            filtered_df = kb_df[mask]
            subset_embeddings = embeddings[filtered_df.index.values]
            index_subset = faiss.IndexFlatIP(subset_embeddings.shape[1])
            index_subset.add(subset_embeddings)
            print(f"      Using filtered index: {len(filtered_df)} interactions")
        else:
            filtered_df = kb_df
            index_subset = index
            print(f"      Using global index")

        distances, indices = index_subset.search(query_embedding, k)
        similarities = distances[0]

        hits = []
        for idx, score in zip(indices[0], similarities):
            if score >= CONFIDENCE_THRESHOLD:
                row = filtered_df.iloc[idx]
                hits.append({
                    "drug1": row['Drug 1'],
                    "drug2": row['Drug 2'],
                    "evidence": row['Interaction Description'],
                    "retrieval_score": float(score)
                })

        if hits:
            print(f"   ✅ Tier 2: Found {len(hits)} semantic matches")
            all_results.append({"query": query_text, "hits": hits, "tier": 2})
        else:
            print(f"   ℹ️ Tier 3: No evidence found")
            all_results.append({"query": query_text, "hits": [], "tier": 3})

    return all_results

## 💊 Polypharmacy Analyzer (N-Drug Pairwise Expansion)

In [9]:
def check_polypharmacy(drug_input):
    """Handle multi-drug queries with pairwise expansion"""
    if isinstance(drug_input, str):
        text = drug_input.replace("interaction", "").strip()
        if " and " in text:
            parts = text.split(" and ")
            first_part = parts[0]
            last_drug = parts[1].strip()
            drugs = [d.strip() for d in first_part.split(",") if d.strip()] + [last_drug]
        else:
            drugs = [d.strip() for d in text.split(",") if d.strip()]
    else:
        drugs = drug_input

    if len(drugs) < 2:
        return {'error': 'Need at least 2 drugs'}

    pairs = list(combinations(drugs, 2))
    queries = [f"{d1} and {d2} interaction" for d1, d2 in pairs]

    print(f"\n🔍 Checking {len(pairs)} drug pairs...")
    results = check_drug_interaction(queries)

    return {'drugs': drugs, 'num_pairs': len(pairs), 'results': results}

In [10]:
check_polypharmacy("Ibuprofen and Warfarin interaction")


🔍 Checking 1 drug pairs...
Tier 1: Direct match for ibuprofen + warfarin


{'drugs': ['Ibuprofen', 'Warfarin'],
 'num_pairs': 1,
 'results': [{'query': 'Ibuprofen and Warfarin interaction',
   'hits': [{'drug1': 'Warfarin',
     'drug2': 'Ibuprofen',
     'evidence': 'Warfarin may increase the anticoagulant activities of Ibuprofen.',
     'retrieval_score': 1.0}],
   'tier': 1}]}

In [11]:
check_polypharmacy("Warfarin, Aspirin and Acetaminophen interaction")


🔍 Checking 3 drug pairs...
Tier 1: Direct match for warfarin + aspirin
Tier 1: Direct match for warfarin + acetaminophen
Tier 1: No direct match, trying semantic search...
Tier 2: Semantic search for aspirin + acetaminophen...
      Using filtered index: 217 interactions
   ✅ Tier 2: Found 5 semantic matches


{'drugs': ['Warfarin', 'Aspirin', 'Acetaminophen'],
 'num_pairs': 3,
 'results': [{'query': 'Warfarin and Aspirin interaction',
   'hits': [{'drug1': 'Warfarin',
     'drug2': 'Acetylsalicylic acid',
     'evidence': 'Warfarin may increase the anticoagulant activities of Acetylsalicylic acid.',
     'retrieval_score': 1.0}],
   'tier': 1},
  {'query': 'Warfarin and Acetaminophen interaction',
   'hits': [{'drug1': 'Warfarin',
     'drug2': 'Acetaminophen',
     'evidence': 'Warfarin may increase the anticoagulant activities of Acetaminophen.',
     'retrieval_score': 1.0}],
   'tier': 1},
  {'query': 'Aspirin and Acetaminophen interaction',
   'hits': [{'drug1': 'Acetaminophen',
     'drug2': 'Ticagrelor',
     'evidence': 'The metabolism of Ticagrelor can be decreased when combined with Acetaminophen.',
     'retrieval_score': 0.6580137014389038},
    {'drug1': 'Acetaminophen',
     'drug2': 'Desipramine',
     'evidence': 'The metabolism of Desipramine can be decreased when combined 

In [12]:
check_polypharmacy("blood thinner and pain medication interaction")


🔍 Checking 1 drug pairs...
   ⚠️ Could not normalize via RxCUI, trying semantic search...
Tier 2: Semantic search for blood thinner + pain...
      Using global index
   ℹ️ Tier 3: No evidence found


{'drugs': ['blood thinner', 'pain medication'],
 'num_pairs': 1,
 'results': [{'query': 'blood thinner and pain medication interaction',
   'hits': [],
   'tier': 3}]}

## 🧪 Pipeline Tests

In [13]:
print("\n" + "=" * 70)
print("PIPELINE VALIDATION")
print("=" * 70)

print("\n1️⃣ Single pair (direct match):")
result = check_drug_interaction("Warfarin and Aspirin interaction")
print(f"   Tier: {result[0]['tier']}, Hits: {len(result[0]['hits'])}")

print("\n2️⃣ Polypharmacy (3 drugs = 3 pairs):")
result = check_polypharmacy("Warfarin, Aspirin and Ibuprofen")
print(f"   Pairs: {result['num_pairs']}, Results: {len(result['results'])}")

print("\n3️⃣ Semantic search fallback:")
result = check_drug_interaction("blood thinner and pain medication interaction")
print(f"   Tier: {result[0]['tier']}, Hits: {len(result[0]['hits'])}")

print("\n✅ Pipeline ready for safety.py integration!")


PIPELINE VALIDATION

1️⃣ Single pair (direct match):
Tier 1: Direct match for warfarin + aspirin
   Tier: 1, Hits: 1

2️⃣ Polypharmacy (3 drugs = 3 pairs):

🔍 Checking 3 drug pairs...
Tier 1: Direct match for warfarin + aspirin
Tier 1: Direct match for warfarin + ibuprofen
Tier 1: Direct match for aspirin + ibuprofen
   Pairs: 3, Results: 3

3️⃣ Semantic search fallback:
   ⚠️ Could not normalize via RxCUI, trying semantic search...
Tier 2: Semantic search for blood thinner + pain...
      Using global index
   ℹ️ Tier 3: No evidence found
   Tier: 3, Hits: 0

✅ Pipeline ready for safety.py integration!
